In [ ]:
from snowflake.snowpark.functions import sproc
from snowflake.snowpark import Session
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.custom_package_usage_config = {
    "enabled": True,
    "cache_path": "@stage1"
}

In [ ]:
# registered SP, passing the module locally (still TODO! try to fix it!)
# must upload secrets.py file in this notebook!
session.add_import("secrets.py")

@sproc(name="showSecrets",
    replace=True,
    is_permanent=True,
    packages=["snowflake-snowpark-python"],
    stage_location="@stage1")
def showSecrets(session: Session) -> str:
    import secrets
    return secrets.show()

session.call("showSecrets")

In [ ]:
# registered SP, passing the module through the stage (still TODO! try to fix it!)
# must upload secrets.py file in this notebook!
session.file.put("secrets.py", "@stage1", auto_compress=False)

@sproc(name="showSecrets2",
    replace=True,
    is_permanent=True,
    packages=["snowflake-snowpark-python"],
    imports=["@stage1/secrets.py"],
    stage_location="@stage1")
def showSecrets2(session: Session) -> str:
    import sys
    import_dir = sys._xoptions.get("snowflake_import_directory")
    sys.path.append(f"{import_dir}secrets.py")
    import secrets
    return secrets.show()

session.call("showSecrets2")